# 05 — Selezione del canale e revisione degli speaker

Inizio dai **185 risultati di diarizzazione** e preparo una sola traccia canonica per ogni registrazione.

Obiettivi:

1. confrontare automaticamente i canali sinistro e destro;
2. scegliere un solo canale per ciascuna registrazione;
3. evitare di trattare i due canali come campioni indipendenti;
4. creare un riepilogo degli speaker sul canale scelto;
5. generare brevi anteprime audio per riconoscere paziente, voce registrata, medico o rumore;
6. produrre una coda di revisione, dando priorità ai casi anomali.

> Le etichette `SPEAKER_00`, `SPEAKER_01`, ecc. sono locali alla singola diarizzazione.  
> Possono quindi essere invertite tra canale sinistro e destro.


In [1]:
from pathlib import Path
import math
import re
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

PROJECT_DIR = Path.cwd()
AUDIO_DIR = PROJECT_DIR / "file_wav"
DIAR_DIR = PROJECT_DIR / "risultati" / "diarizzazione_batch" / "completo"
VALIDATION_DIR = PROJECT_DIR / "risultati" / "validazione_diarizzazione"

OUTPUT_DIR = PROJECT_DIR / "risultati" / "selezione_canale"
PREVIEW_DIR = OUTPUT_DIR / "anteprime_speaker"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("AUDIO_DIR esiste:", AUDIO_DIR.exists())
print("DIAR_DIR esiste:", DIAR_DIR.exists())


PROJECT_DIR: c:\Users\acer\Desktop\ProgettoTesi
AUDIO_DIR esiste: True
DIAR_DIR esiste: True


## 1. Caricamento dei 185 CSV di diarizzazione

Vengono conservate le colonne originali, tra cui:

- `patient_id`
- `recording_id`
- `nome_file`
- `canale`
- `inizio_secondi`
- `fine_secondi`
- `durata_secondi`
- `speaker`


In [2]:
csv_files = sorted(
    p for p in DIAR_DIR.glob("*.csv")
    if p.name.lower() != "stato_elaborazione.csv"
)

if len(csv_files) != 185:
    print(f"[AVVISO] Attesi 185 CSV, trovati {len(csv_files)}.")

tabelle = []

for percorso in csv_files:
    df = pd.read_csv(percorso)

    colonne_richieste = {
        "patient_id", "recording_id", "nome_file", "canale",
        "inizio_secondi", "fine_secondi",
        "durata_secondi", "speaker"
    }

    mancanti = colonne_richieste - set(df.columns)
    if mancanti:
        raise ValueError(
            f"{percorso.name}: colonne mancanti {sorted(mancanti)}"
        )

    df = df.copy()
    df["file_csv"] = percorso.name
    tabelle.append(df)

segmenti = pd.concat(tabelle, ignore_index=True)

for col in ["inizio_secondi", "fine_secondi", "durata_secondi"]:
    segmenti[col] = pd.to_numeric(segmenti[col], errors="coerce")

segmenti = segmenti.dropna(
    subset=["inizio_secondi", "fine_secondi",
            "durata_secondi", "speaker"]
).copy()

segmenti = segmenti[
    (segmenti["durata_secondi"] > 0)
    & (segmenti["fine_secondi"] > segmenti["inizio_secondi"])
].copy()

print("CSV caricati:", len(csv_files))
print("Segmenti caricati:", len(segmenti))
print("Registrazioni:", segmenti["recording_id"].nunique())
print("Combinazioni registrazione-canale:",
      segmenti[["recording_id", "canale"]].drop_duplicates().shape[0])

display(segmenti.head())


CSV caricati: 185
Segmenti caricati: 19722
Registrazioni: 93
Combinazioni registrazione-canale: 185


,patient_id,recording_id,nome_file,canale,inizio_secondi,fine_secondi,durata_secondi,speaker,ruolo,file_csv
0,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,0.368,1.027,0.658,SPEAKER_00,da_definire,ID1.StudioRuggi-006__destro.csv
1,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,1.567,3.338,1.772,SPEAKER_01,da_definire,ID1.StudioRuggi-006__destro.csv
2,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,4.064,5.380,1.316,SPEAKER_00,da_definire,ID1.StudioRuggi-006__destro.csv
3,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,5.903,7.878,1.974,SPEAKER_01,da_definire,ID1.StudioRuggi-006__destro.csv
4,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,6.713,7.439,0.726,SPEAKER_00,da_definire,ID1.StudioRuggi-006__destro.csv


## 2. Funzioni per le metriche di qualità dei canali

Per ciascun canale vengono calcolati:

- durata della registrazione;
- copertura temporale del parlato rilevato;
- RMS complessivo;
- RMS nelle zone di parlato;
- RMS nelle zone senza parlato;
- rapporto segnale/rumore approssimato;
- percentuale di campioni in clipping.

Queste misure non identificano il paziente, ma aiutano a scegliere la traccia più pulita.


In [3]:
def unisci_intervalli(intervalli, limite_massimo=None):
    puliti = []

    for inizio, fine in intervalli:
        inizio = max(0.0, float(inizio))
        fine = float(fine)

        if limite_massimo is not None:
            fine = min(fine, float(limite_massimo))

        if fine > inizio:
            puliti.append((inizio, fine))

    if not puliti:
        return []

    puliti.sort(key=lambda x: x[0])
    uniti = [list(puliti[0])]

    for inizio, fine in puliti[1:]:
        if inizio <= uniti[-1][1]:
            uniti[-1][1] = max(uniti[-1][1], fine)
        else:
            uniti.append([inizio, fine])

    return [(a, b) for a, b in uniti]


def complementari(intervalli_uniti, durata_totale):
    if not intervalli_uniti:
        return [(0.0, durata_totale)]

    gaps = []
    corrente = 0.0

    for inizio, fine in intervalli_uniti:
        if inizio > corrente:
            gaps.append((corrente, inizio))
        corrente = max(corrente, fine)

    if corrente < durata_totale:
        gaps.append((corrente, durata_totale))

    return gaps


def somma_quadrati_intervalli(audio, sample_rate, intervalli):
    somma = 0.0
    conteggio = 0

    for inizio, fine in intervalli:
        i0 = max(0, int(round(inizio * sample_rate)))
        i1 = min(len(audio), int(round(fine * sample_rate)))

        if i1 <= i0:
            continue

        blocco = audio[i0:i1].astype(np.float64, copy=False)
        somma += float(np.dot(blocco, blocco))
        conteggio += len(blocco)

    return somma, conteggio


def dbfs_da_potenza(potenza, eps=1e-12):
    return 10.0 * math.log10(max(float(potenza), eps))


def indice_canale(canale, numero_canali):
    canale = str(canale).lower()

    if numero_canali == 1 or canale == "mono":
        return 0
    if canale == "sinistro":
        return 0
    if canale == "destro":
        return 1

    raise ValueError(f"Canale non riconosciuto: {canale}")


def calcola_metriche_canale(gruppo):
    prima = gruppo.iloc[0]
    nome_file = str(prima["nome_file"])
    percorso_audio = AUDIO_DIR / nome_file

    if not percorso_audio.exists():
        raise FileNotFoundError(percorso_audio)

    audio, sample_rate = sf.read(
        percorso_audio,
        always_2d=True,
        dtype="float32"
    )

    canale = str(prima["canale"]).lower()
    idx = indice_canale(canale, audio.shape[1])
    traccia = audio[:, idx]

    durata_totale = len(traccia) / sample_rate

    intervalli = gruppo[
        ["inizio_secondi", "fine_secondi"]
    ].itertuples(index=False, name=None)

    parlato = unisci_intervalli(
        list(intervalli),
        limite_massimo=durata_totale
    )
    non_parlato = complementari(parlato, durata_totale)

    sq_speech, n_speech = somma_quadrati_intervalli(
        traccia, sample_rate, parlato
    )
    sq_noise, n_noise = somma_quadrati_intervalli(
        traccia, sample_rate, non_parlato
    )

    pot_speech = sq_speech / n_speech if n_speech else np.nan
    pot_noise = sq_noise / n_noise if n_noise else np.nan
    pot_totale = float(np.mean(traccia.astype(np.float64) ** 2))

    if (
        np.isfinite(pot_speech)
        and np.isfinite(pot_noise)
        and pot_noise > 0
    ):
        snr_proxy_db = 10.0 * math.log10(
            max(pot_speech, 1e-12) / max(pot_noise, 1e-12)
        )
    else:
        snr_proxy_db = np.nan

    parlato_secondi = sum(fine - inizio for inizio, fine in parlato)

    return {
        "patient_id": prima["patient_id"],
        "recording_id": prima["recording_id"],
        "nome_file": nome_file,
        "canale": canale,
        "numero_canali_audio": audio.shape[1],
        "sample_rate": sample_rate,
        "durata_audio_secondi": durata_totale,
        "numero_segmenti": len(gruppo),
        "numero_speaker": gruppo["speaker"].nunique(),
        "parlato_unione_secondi": parlato_secondi,
        "copertura_parlato": (
            parlato_secondi / durata_totale
            if durata_totale > 0 else np.nan
        ),
        "rms_totale_dbfs": dbfs_da_potenza(pot_totale),
        "rms_parlato_dbfs": (
            dbfs_da_potenza(pot_speech)
            if np.isfinite(pot_speech) else np.nan
        ),
        "rms_non_parlato_dbfs": (
            dbfs_da_potenza(pot_noise)
            if np.isfinite(pot_noise) else np.nan
        ),
        "snr_proxy_db": snr_proxy_db,
        "clipping_ratio": float(np.mean(np.abs(traccia) >= 0.999)),
    }


## 3. Calcolo delle metriche

Questa cella legge una volta ciascuna combinazione registrazione–canale.  
Può richiedere alcuni minuti.


In [4]:
righe_metriche = []

gruppi = list(segmenti.groupby(["recording_id", "canale"], sort=True))

for indice, ((recording_id, canale), gruppo) in enumerate(gruppi, start=1):
    print(
        f"\r[{indice}/{len(gruppi)}] "
        f"{recording_id} — {canale}",
        end=""
    )

    try:
        riga = calcola_metriche_canale(gruppo)
        riga["errore_metriche"] = ""
    except Exception as exc:
        prima = gruppo.iloc[0]
        riga = {
            "patient_id": prima["patient_id"],
            "recording_id": recording_id,
            "nome_file": prima["nome_file"],
            "canale": canale,
            "numero_segmenti": len(gruppo),
            "numero_speaker": gruppo["speaker"].nunique(),
            "errore_metriche": str(exc),
        }

    righe_metriche.append(riga)

print()

metriche = pd.DataFrame(righe_metriche)

percorso_metriche = OUTPUT_DIR / "metriche_canali.csv"
metriche.to_csv(percorso_metriche, index=False)

print("Righe metriche:", len(metriche))
print("Errori:", (metriche["errore_metriche"].fillna("") != "").sum())
print("Salvato in:", percorso_metriche)

display(metriche.head())


[185/185] ID93.StudioRuggi — sinistrostrooro
Righe metriche: 185
Errori: 0
Salvato in: c:\Users\acer\Desktop\ProgettoTesi\risultati\selezione_canale\metriche_canali.csv


,patient_id,recording_id,nome_file,canale,numero_canali_audio,sample_rate,durata_audio_secondi,numero_segmenti,numero_speaker,parlato_unione_secondi,copertura_parlato,rms_totale_dbfs,rms_parlato_dbfs,rms_non_parlato_dbfs,snr_proxy_db,clipping_ratio,errore_metriche
0,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,2,44100,329.187846,73,2,275.161,0.835878,-30.844290,-30.090914,-45.373879,15.282965,0.0,
1,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,sinistro,2,44100,329.187846,73,2,273.998,0.832345,-34.226706,-33.478481,-45.994298,12.515816,0.0,
2,10,ID10.StudioRuggi,ID10.StudioRuggi.wav,destro,2,48000,203.157333,62,2,136.959,0.674152,-29.189744,-27.653809,-38.318616,10.664807,0.0,
3,10,ID10.StudioRuggi,ID10.StudioRuggi.wav,sinistro,2,48000,203.157333,64,2,136.669,0.672725,-28.013286,-26.422928,-38.424163,12.001234,0.0,
4,101,ID101.StudioRuggi,ID101.StudioRuggi.wav,destro,2,48000,283.648000,99,2,190.766,0.672545,-29.468300,-27.751941,-52.919728,25.167787,0.0,


## 4. Scelta automatica del canale canonico

La regola applicata è:

1. preferire il canale con un numero di speaker più vicino a 2;
2. a parità, preferire il rapporto segnale/rumore approssimato più alto;
3. penalizzare il clipping;
4. usare la copertura del parlato come criterio secondario.

La scelta è un punto di partenza. I casi discordanti restano nella coda di revisione.


In [5]:
def normalizza_serie(serie):
    serie = pd.to_numeric(serie, errors="coerce")
    mediana = serie.median()
    if not np.isfinite(mediana):
        mediana = 0.0

    serie = serie.fillna(mediana)
    minimo = serie.min()
    massimo = serie.max()

    if massimo == minimo:
        return pd.Series(0.0, index=serie.index)

    return (serie - minimo) / (massimo - minimo)


metriche = metriche.copy()

metriche["penalita_numero_speaker"] = (
    metriche["numero_speaker"].sub(2).abs()
)

metriche["snr_norm"] = (
    metriche.groupby("recording_id", group_keys=False)["snr_proxy_db"]
    .apply(normalizza_serie)
)

metriche["coverage_norm"] = (
    metriche.groupby("recording_id", group_keys=False)["copertura_parlato"]
    .apply(normalizza_serie)
)

metriche["clip_norm"] = (
    metriche.groupby("recording_id", group_keys=False)["clipping_ratio"]
    .apply(normalizza_serie)
)

metriche["punteggio_canale"] = (
    -10.0 * metriche["penalita_numero_speaker"]
    + 3.0 * metriche["snr_norm"]
    + 1.0 * metriche["coverage_norm"]
    - 2.0 * metriche["clip_norm"]
)

scelte = []

for recording_id, gruppo in metriche.groupby("recording_id", sort=True):
    gruppo = gruppo.sort_values(
        ["punteggio_canale", "snr_proxy_db", "copertura_parlato"],
        ascending=[False, False, False]
    ).copy()

    scelta = gruppo.iloc[0].copy()

    if len(gruppo) == 1:
        motivo = "registrazione mono"
    else:
        speaker_set = set(gruppo["numero_speaker"].astype(int))

        if len(speaker_set) > 1:
            motivo = (
                "preferito il numero di speaker più vicino a 2; "
                "poi qualità acustica"
            )
        else:
            motivo = (
                "stesso numero di speaker; preferita la qualità "
                "acustica stimata"
            )

    scelta["motivo_scelta"] = motivo
    scelta["discordanza_numero_speaker"] = (
        gruppo["numero_speaker"].nunique() > 1
    )

    if len(gruppo) > 1:
        scelta["margine_punteggio"] = (
            gruppo.iloc[0]["punteggio_canale"]
            - gruppo.iloc[1]["punteggio_canale"]
        )
    else:
        scelta["margine_punteggio"] = np.nan

    scelte.append(scelta)

canali_scelti = pd.DataFrame(scelte).reset_index(drop=True)

percorso_scelte = OUTPUT_DIR / "canale_scelto_per_registrazione.csv"
canali_scelti.to_csv(percorso_scelte, index=False)

print("Registrazioni con canale scelto:", len(canali_scelti))
print("Discordanze tra i canali:",
      int(canali_scelti["discordanza_numero_speaker"].sum()))
print("Salvato in:", percorso_scelte)

display(
    canali_scelti[
        [
            "recording_id", "nome_file", "canale",
            "numero_speaker", "snr_proxy_db",
            "copertura_parlato", "clipping_ratio",
            "discordanza_numero_speaker",
            "motivo_scelta"
        ]
    ].head(20)
)


Registrazioni con canale scelto: 93
Discordanze tra i canali: 5
Salvato in: c:\Users\acer\Desktop\ProgettoTesi\risultati\selezione_canale\canale_scelto_per_registrazione.csv


,recording_id,nome_file,canale,numero_speaker,snr_proxy_db,copertura_parlato,clipping_ratio,discordanza_numero_speaker,motivo_scelta
0,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,2,15.282965,0.835878,0.000000e+00,False,stesso numero di speaker; preferita la qualità...
1,ID10.StudioRuggi,ID10.StudioRuggi.wav,sinistro,2,12.001234,0.672725,0.000000e+00,False,stesso numero di speaker; preferita la qualità...
2,ID101.StudioRuggi,ID101.StudioRuggi.wav,destro,2,25.167787,0.672545,0.000000e+00,False,stesso numero di speaker; preferita la qualità...
3,ID103.StudioRuggi,ID103.StudioRuggi.wav,mono,2,13.504774,0.793758,0.000000e+00,False,registrazione mono
4,ID104.StudioRuggi,ID104.StudioRuggi.wav,sinistro,2,24.078922,0.798363,0.000000e+00,False,stesso numero di speaker; preferita la qualità...
5,ID108.StudioRuggi,ID108.StudioRuggi.wav,destro,2,24.445137,0.402510,0.000000e+00,True,preferito il numero di speaker più vicino a 2;...
6,ID11.StudioRuggi,ID11.StudioRuggi.wav,destro,2,7.970024,0.534317,0.000000e+00,False,stesso numero di speaker; preferita la qualità...
7,ID112.StudioRuggi,ID112.StudioRuggi.wav,sinistro,3,25.865249,0.716599,1.119013e-07,False,stesso numero di speaker; preferita la qualità...
8,ID114.StudioRuggi,ID114.StudioRuggi.wav,sinistro,2,24.431470,0.694100,0.000000e+00,False,stesso numero di speaker; preferita la qualità...
9,ID116.StudioRuggi,ID116.StudioRuggi.wav,sinistro,2,9.040408,0.576537,0.000000e+00,False,stesso numero di speaker; preferita la qualità...


## 5. Riepilogo degli speaker sul solo canale scelto

Da questo punto in poi ogni registrazione compare una sola volta.

La colonna `speaker_dominante` segnala soltanto lo speaker con maggiore durata totale.  
Non equivale automaticamente al paziente.


In [6]:
chiavi_scelte = canali_scelti[
    ["recording_id", "canale"]
].drop_duplicates()

segmenti_scelti = segmenti.merge(
    chiavi_scelte,
    on=["recording_id", "canale"],
    how="inner"
)

riepilogo_speaker = (
    segmenti_scelti
    .groupby(
        ["patient_id", "recording_id", "nome_file",
         "canale", "speaker"],
        as_index=False
    )
    .agg(
        numero_segmenti=("speaker", "size"),
        durata_totale_secondi=("durata_secondi", "sum"),
        durata_media_secondi=("durata_secondi", "mean"),
        durata_massima_secondi=("durata_secondi", "max"),
    )
)

riepilogo_speaker["quota_parlato_nella_registrazione"] = (
    riepilogo_speaker["durata_totale_secondi"]
    / riepilogo_speaker.groupby(
        "recording_id"
    )["durata_totale_secondi"].transform("sum")
)

idx_dominante = (
    riepilogo_speaker
    .groupby("recording_id")["durata_totale_secondi"]
    .idxmax()
)

riepilogo_speaker["speaker_dominante"] = False
riepilogo_speaker.loc[idx_dominante, "speaker_dominante"] = True

numero_speaker_scelto = (
    riepilogo_speaker
    .groupby("recording_id")["speaker"]
    .nunique()
    .rename("numero_speaker_canale_scelto")
)

riepilogo_speaker = riepilogo_speaker.merge(
    numero_speaker_scelto,
    on="recording_id",
    how="left"
)

riepilogo_speaker = riepilogo_speaker.merge(
    canali_scelti[
        [
            "recording_id",
            "discordanza_numero_speaker",
            "margine_punteggio"
        ]
    ],
    on="recording_id",
    how="left"
)

riepilogo_speaker["priorita_revisione"] = np.select(
    [
        riepilogo_speaker["numero_speaker_canale_scelto"] > 2,
        riepilogo_speaker["numero_speaker_canale_scelto"] < 2,
        riepilogo_speaker["discordanza_numero_speaker"],
    ],
    [
        "alta: più di 2 speaker",
        "alta: meno di 2 speaker",
        "media: canali discordanti",
    ],
    default="ordinaria"
)

riepilogo_speaker["ruolo_manual"] = ""
riepilogo_speaker["confidenza_manual"] = ""
riepilogo_speaker["note_manual"] = ""

percorso_riepilogo = OUTPUT_DIR / "riepilogo_speaker_canale_scelto.csv"
riepilogo_speaker.to_csv(percorso_riepilogo, index=False)

print("Speaker da revisionare:", len(riepilogo_speaker))
print("Registrazioni:", riepilogo_speaker["recording_id"].nunique())
print("Salvato in:", percorso_riepilogo)

display(
    riepilogo_speaker.sort_values(
        ["priorita_revisione", "recording_id",
         "durata_totale_secondi"],
        ascending=[True, True, False]
    ).head(30)
)


Speaker da revisionare: 197
Registrazioni: 93
Salvato in: c:\Users\acer\Desktop\ProgettoTesi\risultati\selezione_canale\riepilogo_speaker_canale_scelto.csv


,patient_id,recording_id,nome_file,canale,speaker,numero_segmenti,durata_totale_secondi,durata_media_secondi,durata_massima_secondi,quota_parlato_nella_registrazione,speaker_dominante,numero_speaker_canale_scelto,discordanza_numero_speaker,margine_punteggio,priorita_revisione,ruolo_manual,confidenza_manual,note_manual
153,112,ID112.StudioRuggi,ID112.StudioRuggi.wav,sinistro,SPEAKER_00,61,137.023,2.246279,8.623,0.495543,True,3,False,2.0,alta: più di 2 speaker,,,
154,112,ID112.StudioRuggi,ID112.StudioRuggi.wav,sinistro,SPEAKER_01,49,121.802,2.485755,11.492,0.440496,False,3,False,2.0,alta: più di 2 speaker,,,
155,112,ID112.StudioRuggi,ID112.StudioRuggi.wav,sinistro,SPEAKER_02,6,17.686,2.947667,6.345,0.063961,False,3,False,2.0,alta: più di 2 speaker,,,
162,118,ID118.StudioRuggi,ID118.StudioRuggi.wav,sinistro,SPEAKER_00,59,142.379,2.413203,17.246,0.715167,True,3,False,2.0,alta: più di 2 speaker,,,
163,118,ID118.StudioRuggi,ID118.StudioRuggi.wav,sinistro,SPEAKER_01,42,47.576,1.132762,6.328,0.238973,False,3,False,2.0,alta: più di 2 speaker,,,
164,118,ID118.StudioRuggi,ID118.StudioRuggi.wav,sinistro,SPEAKER_02,10,9.130,0.913000,1.721,0.045860,False,3,False,2.0,alta: più di 2 speaker,,,
167,121,ID121.StudioRuggi,ID121.StudioRuggi.wav,sinistro,SPEAKER_02,63,135.562,2.151778,8.573,0.684411,True,3,False,4.0,alta: più di 2 speaker,,,
165,121,ID121.StudioRuggi,ID121.StudioRuggi.wav,sinistro,SPEAKER_00,45,47.454,1.054533,2.936,0.239581,False,3,False,4.0,alta: più di 2 speaker,,,
166,121,ID121.StudioRuggi,ID121.StudioRuggi.wav,sinistro,SPEAKER_01,13,15.055,1.158077,3.544,0.076008,False,3,False,4.0,alta: più di 2 speaker,,,
189,155,ID155.StudioRuggi,ID155.StudioRuggi.wav,destro,SPEAKER_01,63,135.655,2.153254,8.657,0.497986,True,3,False,2.0,alta: più di 2 speaker,,,


## 6. Creazione delle anteprime audio

Per ciascuno speaker vengono concatenati alcuni dei segmenti più lunghi, fino a circa 25 secondi complessivi.

Le anteprime servono soltanto per l'ascolto e la revisione manuale.  
Non verranno usate come input del modello.


In [7]:
def nome_sicuro(testo):
    testo = re.sub(r"[^A-Za-z0-9._-]+", "_", str(testo))
    return testo.strip("_")


def crea_anteprima_speaker(
    gruppo,
    durata_massima_totale=25.0,
    durata_massima_segmento=10.0,
    pausa_secondi=0.25
):
    prima = gruppo.iloc[0]
    percorso_audio = AUDIO_DIR / str(prima["nome_file"])

    audio, sample_rate = sf.read(
        percorso_audio,
        always_2d=True,
        dtype="float32"
    )

    idx = indice_canale(
        prima["canale"],
        audio.shape[1]
    )
    traccia = audio[:, idx]

    selezione = gruppo.sort_values(
        "durata_secondi",
        ascending=False
    )

    blocchi = []
    durata_accumulata = 0.0
    silenzio = np.zeros(
        int(round(pausa_secondi * sample_rate)),
        dtype=np.float32
    )

    for _, riga in selezione.iterrows():
        durata_disponibile = (
            durata_massima_totale - durata_accumulata
        )

        if durata_disponibile <= 0:
            break

        durata_uso = min(
            float(riga["durata_secondi"]),
            durata_massima_segmento,
            durata_disponibile
        )

        i0 = max(
            0,
            int(round(float(riga["inizio_secondi"]) * sample_rate))
        )
        i1 = min(
            len(traccia),
            i0 + int(round(durata_uso * sample_rate))
        )

        if i1 <= i0:
            continue

        blocco = traccia[i0:i1]

        if blocchi:
            blocchi.append(silenzio)

        blocchi.append(blocco)
        durata_accumulata += len(blocco) / sample_rate

    if not blocchi:
        return None

    anteprima = np.concatenate(blocchi)

    nome_output = (
        f"{nome_sicuro(prima['recording_id'])}"
        f"__{nome_sicuro(prima['canale'])}"
        f"__{nome_sicuro(prima['speaker'])}.wav"
    )

    percorso_output = PREVIEW_DIR / nome_output
    sf.write(percorso_output, anteprima, sample_rate)

    return {
        "percorso_anteprima": str(percorso_output),
        "nome_anteprima": nome_output,
        "durata_anteprima_secondi": len(anteprima) / sample_rate,
    }


anteprime = []

gruppi_speaker = list(
    segmenti_scelti.groupby(
        ["recording_id", "canale", "speaker"],
        sort=True
    )
)

for indice, (_, gruppo) in enumerate(gruppi_speaker, start=1):
    prima = gruppo.iloc[0]

    print(
        f"\r[{indice}/{len(gruppi_speaker)}] "
        f"{prima['recording_id']} — "
        f"{prima['canale']} — "
        f"{prima['speaker']}",
        end=""
    )

    try:
        risultato = crea_anteprima_speaker(gruppo)

        if risultato is not None:
            risultato.update({
                "recording_id": prima["recording_id"],
                "canale": prima["canale"],
                "speaker": prima["speaker"],
            })
            anteprime.append(risultato)
    except Exception as exc:
        print(
            f"\n[AVVISO] {prima['recording_id']} "
            f"{prima['speaker']}: {exc}"
        )

print()

df_anteprime = pd.DataFrame(anteprime)

riepilogo_revisione = riepilogo_speaker.merge(
    df_anteprime,
    on=["recording_id", "canale", "speaker"],
    how="left"
)

percorso_revisione = OUTPUT_DIR / "coda_revisione_speaker.csv"

if not percorso_revisione.exists():
    riepilogo_revisione.to_csv(percorso_revisione, index=False)
    print("Coda di revisione creata in:", percorso_revisione)
else:
    print(
        "La coda esiste già e non è stata sovrascritta:",
        percorso_revisione
    )

print("Anteprime create:", len(df_anteprime))
print("Cartella anteprime:", PREVIEW_DIR)


[197/197] ID93.StudioRuggi — sinistro — SPEAKER_01011_01
Coda di revisione creata in: c:\Users\acer\Desktop\ProgettoTesi\risultati\selezione_canale\coda_revisione_speaker.csv
Anteprime create: 197
Cartella anteprime: c:\Users\acer\Desktop\ProgettoTesi\risultati\selezione_canale\anteprime_speaker


## 7. Coda prioritaria

Si parte da:

1. registrazioni con più di 2 speaker sul canale scelto;
2. registrazioni con meno di 2 speaker;
3. registrazioni in cui i due canali non concordano;
4. un piccolo campione delle registrazioni ordinarie.


In [8]:
ordine_priorita = {
    "alta: più di 2 speaker": 0,
    "alta: meno di 2 speaker": 1,
    "media: canali discordanti": 2,
    "ordinaria": 3,
}

coda = riepilogo_revisione.copy()
coda["ordine"] = coda["priorita_revisione"].map(
    ordine_priorita
).fillna(99)

coda = coda.sort_values(
    [
        "ordine",
        "recording_id",
        "durata_totale_secondi"
    ],
    ascending=[True, True, False]
).reset_index(drop=True)

display(
    coda[
        [
            "recording_id", "canale", "speaker",
            "numero_speaker_canale_scelto",
            "durata_totale_secondi",
            "quota_parlato_nella_registrazione",
            "speaker_dominante",
            "priorita_revisione",
            "nome_anteprima"
        ]
    ].head(50)
)


,recording_id,canale,speaker,numero_speaker_canale_scelto,durata_totale_secondi,quota_parlato_nella_registrazione,speaker_dominante,priorita_revisione,nome_anteprima
0,ID112.StudioRuggi,sinistro,SPEAKER_00,3,137.023,0.495543,True,alta: più di 2 speaker,ID112.StudioRuggi__sinistro__SPEAKER_00.wav
1,ID112.StudioRuggi,sinistro,SPEAKER_01,3,121.802,0.440496,False,alta: più di 2 speaker,ID112.StudioRuggi__sinistro__SPEAKER_01.wav
2,ID112.StudioRuggi,sinistro,SPEAKER_02,3,17.686,0.063961,False,alta: più di 2 speaker,ID112.StudioRuggi__sinistro__SPEAKER_02.wav
3,ID118.StudioRuggi,sinistro,SPEAKER_00,3,142.379,0.715167,True,alta: più di 2 speaker,ID118.StudioRuggi__sinistro__SPEAKER_00.wav
4,ID118.StudioRuggi,sinistro,SPEAKER_01,3,47.576,0.238973,False,alta: più di 2 speaker,ID118.StudioRuggi__sinistro__SPEAKER_01.wav
5,ID118.StudioRuggi,sinistro,SPEAKER_02,3,9.130,0.045860,False,alta: più di 2 speaker,ID118.StudioRuggi__sinistro__SPEAKER_02.wav
6,ID121.StudioRuggi,sinistro,SPEAKER_02,3,135.562,0.684411,True,alta: più di 2 speaker,ID121.StudioRuggi__sinistro__SPEAKER_02.wav
7,ID121.StudioRuggi,sinistro,SPEAKER_00,3,47.454,0.239581,False,alta: più di 2 speaker,ID121.StudioRuggi__sinistro__SPEAKER_00.wav
8,ID121.StudioRuggi,sinistro,SPEAKER_01,3,15.055,0.076008,False,alta: più di 2 speaker,ID121.StudioRuggi__sinistro__SPEAKER_01.wav
9,ID155.StudioRuggi,destro,SPEAKER_01,3,135.655,0.497986,True,alta: più di 2 speaker,ID155.StudioRuggi__destro__SPEAKER_01.wav


## 8. Ascolto delle anteprime

Usare `ascolta_anteprima` passando registrazione e speaker.

Dopo l'ascolto, compilare nel file:

```text
risultati/selezione_canale/coda_revisione_speaker.csv
```

le colonne:

- `ruolo_manual`
- `confidenza_manual`
- `note_manual`

Valori consigliati per `ruolo_manual`:

- `paziente`
- `voce_registrata`
- `medico`
- `terza_persona`
- `rumore`
- `incerto`


In [9]:
def ascolta_anteprima(recording_id, speaker):
    selezione = coda[
        (coda["recording_id"].astype(str) == str(recording_id))
        & (coda["speaker"].astype(str) == str(speaker))
    ]

    if selezione.empty:
        print("Anteprima non trovata.")
        return

    for _, riga in selezione.iterrows():
        percorso = Path(riga["percorso_anteprima"])

        print(
            f"{riga['recording_id']} | "
            f"{riga['canale']} | "
            f"{riga['speaker']} | "
            f"{riga['priorita_revisione']}"
        )

        if percorso.exists():
            display(Audio(filename=str(percorso)))
        else:
            print("File non trovato:", percorso)


# Esempio:
# ascolta_anteprima("ID30.StudioRuggi", "SPEAKER_00")


## 9. Controllo finale delle scelte

Questa tabella mostra soltanto le registrazioni considerate anomale o discordanti.


In [10]:
anomale = canali_scelti[
    (canali_scelti["numero_speaker"] != 2)
    | (canali_scelti["discordanza_numero_speaker"])
].copy()

display(
    anomale[
        [
            "recording_id", "nome_file", "canale",
            "numero_speaker", "snr_proxy_db",
            "copertura_parlato",
            "discordanza_numero_speaker",
            "margine_punteggio",
            "motivo_scelta"
        ]
    ].sort_values("recording_id")
)

print("Registrazioni da controllare prioritariamente:",
      anomale["recording_id"].nunique())


,recording_id,nome_file,canale,numero_speaker,snr_proxy_db,copertura_parlato,discordanza_numero_speaker,margine_punteggio,motivo_scelta
5,ID108.StudioRuggi,ID108.StudioRuggi.wav,destro,2,24.445137,0.402510,True,14.0,preferito il numero di speaker più vicino a 2;...
7,ID112.StudioRuggi,ID112.StudioRuggi.wav,sinistro,3,25.865249,0.716599,False,2.0,stesso numero di speaker; preferita la qualità...
11,ID118.StudioRuggi,ID118.StudioRuggi.wav,sinistro,3,10.732036,0.474874,False,2.0,stesso numero di speaker; preferita la qualità...
13,ID121.StudioRuggi,ID121.StudioRuggi.wav,sinistro,3,20.640038,0.669161,False,4.0,stesso numero di speaker; preferita la qualità...
27,ID155.StudioRuggi,ID155.StudioRuggi.wav,destro,3,16.318453,0.601012,False,2.0,stesso numero di speaker; preferita la qualità...
43,ID29.StudioRuggi,ID29.StudioRuggi.wav,sinistro,2,10.765208,0.692368,True,6.0,preferito il numero di speaker più vicino a 2;...
45,ID30.StudioRuggi,ID30.StudioRuggi.wav,sinistro,4,13.411051,0.583720,False,4.0,stesso numero di speaker; preferita la qualità...
51,ID37.Studio-Ruggi-014,ID37.Studio-Ruggi-014.wav,destro,2,11.590802,0.599788,True,14.0,preferito il numero di speaker più vicino a 2;...
65,ID53.StudioRuggi,ID53.StudioRuggi.wav,sinistro,3,19.363664,0.639968,False,4.0,stesso numero di speaker; preferita la qualità...
76,ID68.StudioRuggi,ID68.StudioRuggi.wav,destro,3,12.754186,0.670421,False,2.0,stesso numero di speaker; preferita la qualità...


Registrazioni da controllare prioritariamente: 15
